In [1]:
# ✅ Cell 1 — Upload ZIP/CSV, stage as DATA_CSV, and preview 5 head + 5 tail
import os, io, zipfile, shutil
from pathlib import Path
import pandas as pd

# === Setup working dirs (Colab-friendly; falls back if not in Colab)
def get_base_dir():
    for p in [Path("/content"), Path("/mnt/data")]:
        try:
            test = p / "_write_test"
            test.write_text("ok", encoding="utf-8")
            test.unlink(missing_ok=True)
            return p
        except Exception:
            continue
    return Path.cwd()

BASE_DIR = get_base_dir()
RAW_DIR  = BASE_DIR / "data_raw"
OUT_DIR  = BASE_DIR / "data"
RAW_DIR.mkdir(exist_ok=True, parents=True)
OUT_DIR.mkdir(exist_ok=True, parents=True)

# --- Colab upload helper
def _upload():
    try:
        from google.colab import files
        print("📤 Upload your .zip (preferred) or .csv file…")
        up = files.upload()
        if not up:
            raise RuntimeError("No file uploaded.")
        fname = list(up.keys())[0]
        with open(RAW_DIR / fname, "wb") as f:
            f.write(up[fname])
        print(f"✅ Uploaded: {fname} ({len(up[fname]):,} bytes)")
        return str(RAW_DIR / fname)
    except Exception:
        print("ℹ️ If not in Colab, place your file under:", RAW_DIR)
        return None

uploaded_path = _upload()

# --- Stage the main CSV
def stage_csv(uploaded_path: str | None) -> str | None:
    if uploaded_path is None:
        return None
    p = Path(uploaded_path)
    if p.suffix.lower() == ".csv":
        staged = OUT_DIR / "dataset.csv"
        if staged.exists():
            staged.unlink()
        shutil.copy2(p, staged)
        return str(staged)
    if p.suffix.lower() == ".zip":
        with zipfile.ZipFile(p, 'r') as zf:
            zf.extractall(RAW_DIR / "unzipped")
        csvs = list((RAW_DIR / "unzipped").rglob("*.csv"))
        if not csvs:
            raise RuntimeError("No CSV files found in ZIP.")
        best = max(csvs, key=lambda x: x.stat().st_size)
        staged = OUT_DIR / "dataset.csv"
        if staged.exists():
            staged.unlink()
        shutil.copy2(best, staged)
        print(f"📦 Found {len(csvs)} CSV(s) in ZIP. Using largest: {best.name} ({best.stat().st_size:,} bytes)")
        return str(staged)
    raise RuntimeError("Unsupported file type. Please upload .zip or .csv")

DATA_CSV = stage_csv(uploaded_path)

# --- Helpers to preview head/tail efficiently
def read_csv_head(path, n=5):
    return pd.read_csv(path, nrows=n, dtype=str, engine="python", on_bad_lines="skip")

def read_csv_tail(path, n=5, encoding="utf-8"):
    # Read last N data lines without loading whole file
    header = None
    with open(path, "r", encoding=encoding, errors="ignore") as f:
        header = f.readline().rstrip("\n")
    # Read from end in binary chunks until we have >= n non-empty lines
    chunk_size = 256 * 1024
    data = b""
    with open(path, "rb") as fb:
        fb.seek(0, os.SEEK_END)
        file_size = fb.tell()
        pos = file_size
        lines = []
        while pos > 0 and len(lines) <= n:
            read_size = min(chunk_size, pos)
            fb.seek(pos - read_size, os.SEEK_SET)
            data = fb.read(read_size) + data
            pos -= read_size
            lines = data.split(b"\n")
        # Last element may be partial; take last n complete lines ignoring header
        # Decode safely and filter out empty lines
        decoded = [ln.decode(encoding, errors="ignore") for ln in lines]
        # Drop possible duplicated header if we read from the beginning
        tail_lines = [ln for ln in decoded if ln.strip()]
        # Remove header line if present among tail slice
        tail_data = [ln for ln in tail_lines if ln.strip() != header]
        tail_last_n = tail_data[-n:] if len(tail_data) >= n else tail_data
        csv_buf = io.StringIO(header + "\n" + "\n".join(tail_last_n))
        return pd.read_csv(csv_buf, dtype=str, engine="python", on_bad_lines="skip")

if DATA_CSV:
    file_size = Path(DATA_CSV).stat().st_size
    print(f"\n✅ Staged CSV: {DATA_CSV}  ({file_size:,} bytes)")
    head5 = read_csv_head(DATA_CSV, 5)
    tail5 = read_csv_tail(DATA_CSV, 5)
    print("\n🔹 Head (5 rows):")
    display(head5)
    print("\n🔹 Tail (5 rows):")
    display(tail5)
    print("\n🧱 Columns detected:", list(head5.columns))
else:
    print(f"📂 Put your ZIP/CSV under: {RAW_DIR} and rerun this cell.")


📤 Upload your .zip (preferred) or .csv file…


Saving merged (2)http.zip to merged (2)http.zip
✅ Uploaded: merged (2)http.zip (10,437,175 bytes)
📦 Found 1 CSV(s) in ZIP. Using largest: merged (2).csv (212,971,800 bytes)

✅ Staged CSV: /content/data/dataset.csv  (212,971,800 bytes)

🔹 Head (5 rows):


,@timestamp,http.request.method,http.response.status_code,http.version,http.response.body.bytes,user_agent.original,url.path,url.domain,destination.ip,destination.port,source.ip
0,"Sep 20, 2025 @ 11:59:59.895",POST,200,1.1,88,Mozilla/5.0,/api,149.154.167.91,172.16.60.2,"8,080",10.10.10.69
1,"Sep 20, 2025 @ 11:59:59.866",POST,200,1.1,88,Mozilla/5.0,/api,149.154.167.92,172.16.60.2,"8,080",192.168.10.11
2,"Sep 20, 2025 @ 11:59:59.816",POST,200,1.1,88,Mozilla/5.0,/api,149.154.167.92,172.16.60.2,"8,080",192.168.10.13
3,"Sep 20, 2025 @ 11:59:59.802",POST,200,1.1,88,Mozilla/5.0,/api,149.154.167.92,172.16.60.2,"8,080",192.168.10.13
4,"Sep 20, 2025 @ 11:59:59.588",POST,200,1.1,88,Mozilla/5.0,/api,149.154.167.91,172.16.60.2,"8,080",10.10.10.69



🔹 Tail (5 rows):


,@timestamp,http.request.method,http.response.status_code,http.version,http.response.body.bytes,user_agent.original,url.path,url.domain,destination.ip,destination.port,source.ip
0,"Oct 7, 2025 @ 09:08:03.073",POST,-,-,0,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,/,speed.cloudflare.com,213.111.144.5,80,10.10.10.60
1,"Oct 7, 2025 @ 09:08:02.998",POST,-,-,0,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,/,speed.cloudflare.com,213.111.144.5,80,10.10.10.60
2,"Oct 7, 2025 @ 09:08:02.998",POST,-,-,0,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,/,speed.cloudflare.com,213.111.144.5,80,10.10.10.60
3,"Oct 7, 2025 @ 09:08:02.997",POST,-,-,0,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,/,speed.cloudflare.com,213.111.144.5,80,10.10.10.60
4,"Oct 7, 2025 @ 09:08:02.997",POST,-,-,0,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,/,speed.cloudflare.com,213.111.144.5,80,10.10.10.60



🧱 Columns detected: ['@timestamp', 'http.request.method', 'http.response.status_code', 'http.version', 'http.response.body.bytes', 'user_agent.original', 'url.path', 'url.domain', 'destination.ip', 'destination.port', 'source.ip']


In [2]:
# ✅ Cell 2 — Build features (status_class, uri_len, ua_group), light cleaning, preview head/tail (5)
import re
import pandas as pd

assert 'DATA_CSV' in globals() and DATA_CSV is not None, "DATA_CSV not set. Run Cell 1 first."

# --- Read full CSV (all 11 columns as strings; we will type-cast selected ones)
df = pd.read_csv(DATA_CSV, dtype=str, engine="python", on_bad_lines="skip")

# Ensure columns exist (exact names from your sample)
expected_cols = [
    "@timestamp",
    "http.request.method",
    "http.response.status_code",
    "http.version",
    "http.response.body.bytes",
    "user_agent.original",
    "url.path",
    "url.domain",
    "destination.ip",
    "destination.port",
    "source.ip",
]
missing = [c for c in expected_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing expected columns: {missing}")

# --- Basic cleaning helpers
def to_int_series(s: pd.Series, default=0):
    # keep digits only (handles things like "8,080")
    cleaned = s.fillna("").astype(str).str.replace(r"[^\d]", "", regex=True)
    out = pd.to_numeric(cleaned, errors="coerce").fillna(default).astype("int64")
    return out

# Numeric cleanups
df["status_code_num"] = to_int_series(df["http.response.status_code"])
df["response_bytes_num"] = to_int_series(df["http.response.body.bytes"])
df["dest_port_num"] = to_int_series(df["destination.port"])

# --- Feature 1: status_class (2/3/4/5)
df["status_class"] = (df["status_code_num"] // 100).astype("int16")

# --- Feature 2: uri_len
df["uri_len"] = df["url.path"].fillna("").astype(str).str.len().astype("int32")

# --- Feature 3: ua_group (browser / bot / empty / other)
ua = df["user_agent.original"].fillna("").astype(str)
ua_l = ua.str.lower()
is_browser = ua_l.str.contains(r"mozilla|chrome|safari|firefox|edge", regex=True)
is_bot = ua_l.str.contains(r"curl|python|bot|go-http-client|wget|java|requests", regex=True)
is_empty = ua_l.str.strip().eq("")

df["ua_group"] = (
    is_browser.map({True: "browser", False: None})
    .fillna(is_bot.map({True: "bot", False: None}))
    .fillna(is_empty.map({True: "empty", False: "other"}))
)

# --- Keep original 11 columns + the 3 new ones (total 14)
final_cols = expected_cols + ["status_class", "uri_len", "ua_group"]
df_out = df[final_cols].copy()

# --- Preview head/tail (5)
print("✅ Features built: status_class, uri_len, ua_group")
print("🧱 Columns:", list(df_out.columns))
print("\n🔹 Head (5 rows):")
display(df_out.head(5))
print("\n🔹 Tail (5 rows):")
display(df_out.tail(5))


✅ Features built: status_class, uri_len, ua_group
🧱 Columns: ['@timestamp', 'http.request.method', 'http.response.status_code', 'http.version', 'http.response.body.bytes', 'user_agent.original', 'url.path', 'url.domain', 'destination.ip', 'destination.port', 'source.ip', 'status_class', 'uri_len', 'ua_group']

🔹 Head (5 rows):


,@timestamp,http.request.method,http.response.status_code,http.version,http.response.body.bytes,user_agent.original,url.path,url.domain,destination.ip,destination.port,source.ip,status_class,uri_len,ua_group
0,"Sep 20, 2025 @ 11:59:59.895",POST,200,1.1,88,Mozilla/5.0,/api,149.154.167.91,172.16.60.2,"8,080",10.10.10.69,2,4,browser
1,"Sep 20, 2025 @ 11:59:59.866",POST,200,1.1,88,Mozilla/5.0,/api,149.154.167.92,172.16.60.2,"8,080",192.168.10.11,2,4,browser
2,"Sep 20, 2025 @ 11:59:59.816",POST,200,1.1,88,Mozilla/5.0,/api,149.154.167.92,172.16.60.2,"8,080",192.168.10.13,2,4,browser
3,"Sep 20, 2025 @ 11:59:59.802",POST,200,1.1,88,Mozilla/5.0,/api,149.154.167.92,172.16.60.2,"8,080",192.168.10.13,2,4,browser
4,"Sep 20, 2025 @ 11:59:59.588",POST,200,1.1,88,Mozilla/5.0,/api,149.154.167.91,172.16.60.2,"8,080",10.10.10.69,2,4,browser



🔹 Tail (5 rows):


,@timestamp,http.request.method,http.response.status_code,http.version,http.response.body.bytes,user_agent.original,url.path,url.domain,destination.ip,destination.port,source.ip,status_class,uri_len,ua_group
1719208,"Oct 7, 2025 @ 09:08:03.073",POST,-,-,0,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,/,speed.cloudflare.com,213.111.144.5,80,10.10.10.60,0,1,browser
1719209,"Oct 7, 2025 @ 09:08:02.998",POST,-,-,0,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,/,speed.cloudflare.com,213.111.144.5,80,10.10.10.60,0,1,browser
1719210,"Oct 7, 2025 @ 09:08:02.998",POST,-,-,0,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,/,speed.cloudflare.com,213.111.144.5,80,10.10.10.60,0,1,browser
1719211,"Oct 7, 2025 @ 09:08:02.997",POST,-,-,0,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,/,speed.cloudflare.com,213.111.144.5,80,10.10.10.60,0,1,browser
1719212,"Oct 7, 2025 @ 09:08:02.997",POST,-,-,0,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,/,speed.cloudflare.com,213.111.144.5,80,10.10.10.60,0,1,browser


In [9]:
# 🧱 Cell 3 — Build feature matrix X (with path-content & rate features)
# - Fast timestamp parsing
# - Path-content IOC flags + path depth
# - Per-IP request rates (1s, 5s buckets)
# - OneHot on categoricals, RobustScale on numerics
# - Down-weight ALL path-derived features + response_bytes to 1/3
# - Save artifacts compatible with Cells 4–7

!pip -q install pandas numpy scikit-learn pyarrow joblib

import os, re, json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, RobustScaler
import joblib

assert 'DATA_CSV' in globals() and DATA_CSV is not None, "Run Cell 1 first to set DATA_CSV."

# === Read full CSV (strings), we'll convert selected cols
df = pd.read_csv(DATA_CSV, dtype=str, engine="python", on_bad_lines="skip")

# keep exactly your 11 base columns
BASE_COLS = [
    "@timestamp",
    "http.request.method",
    "http.response.status_code",
    "http.version",
    "http.response.body.bytes",
    "user_agent.original",
    "url.path",
    "url.domain",
    "destination.ip",
    "destination.port",
    "source.ip",
]
missing = [c for c in BASE_COLS if c not in df.columns]
if missing:
    raise ValueError(f"Missing expected columns: {missing}")

# -----------------------------
# 1) Fast timestamp parse (2 formats)
# -----------------------------
ts_raw = df["@timestamp"].fillna("").astype(str)

# common formats seen in your sample:
# "Oct 19, 2025 @ 15:16:52.151"  -> "%b %d, %Y @ %H:%M:%S.%f"
# "Oct 19, 2025 @ 15:16:52"      -> "%b %d, %Y @ %H:%M:%S"
def parse_ts_fast(s: pd.Series) -> pd.Series:
    s1 = pd.to_datetime(s, format="%b %d, %Y @ %H:%M:%S.%f", errors="coerce", utc=True)
    mask = s1.isna()
    if mask.any():
        s2 = pd.to_datetime(s[mask], format="%b %d, %Y @ %H:%M:%S", errors="coerce", utc=True)
        s1.loc[mask] = s2
    return s1

ts = parse_ts_fast(ts_raw)
# Fallback (rare): still NaT -> keep as is; we'll handle downstream
sec_bucket = (ts.view('int64') // 10**9).astype('Int64')  # epoch seconds (nullable)
bucket5    = ((ts.view('int64') // 10**9) // 5).astype('Int64')

# -----------------------------
# 2) Helpers & base numerics
# -----------------------------
def to_int(s: pd.Series, default=0):
    return pd.to_numeric(s.fillna("").astype(str).str.replace(r"[^\d]", "", regex=True),
                         errors="coerce").fillna(default).astype("int64")

def http_version_float(s: pd.Series):
    return pd.to_numeric(s.fillna("").str.extract(r"(\d+(?:\.\d+)?)", expand=False),
                         errors="coerce").fillna(0.0).astype("float32")

def ua_group_from(ua: pd.Series) -> pd.Series:
    u = ua.fillna("").astype(str).str.lower()
    is_browser = u.str.contains(r"mozilla|chrome|safari|firefox|edge")
    is_bot     = u.str.contains(r"curl|python|bot|go-http-client|wget|java|requests|okhttp|axios|postmanruntime|telegrambot|slackbot|discord|libwww-perl")
    is_empty   = u.str.strip().eq("")
    return (
        is_browser.map({True: "browser", False: None})
        .fillna(is_bot.map({True: "bot", False: None}))
        .fillna(is_empty.map({True: "empty", False: "other"}))
    )

# Base numeric features
status_code_num  = to_int(df["http.response.status_code"])
status_class     = (status_code_num // 100).astype("int16")
response_bytes   = to_int(df["http.response.body.bytes"])         # <-- will be down-weighted
dest_port_num    = to_int(df["destination.port"])
ver              = http_version_float(df["http.version"])
uri_len          = df["url.path"].fillna("").astype(str).str.len().astype("int32")  # path-derived (down-weight)

# -----------------------------
# 3) Path-content features (IOC-like) — all considered "path-derived"
# -----------------------------
path = df["url.path"].fillna("").astype(str)

# normalize path a bit for regex checks
path_l = path.str.lower()

is_config_like   = path_l.str.contains(r"(?:^|/)(?:config|configuration)(?:[.\-_]|/|$)")
is_settings_like = path_l.str.contains(r"(?:^|/)(?:setting|settings)(?:[.\-_]|/|$)")
is_db_like       = path_l.str.contains(r"(?:^|/)(?:db|database)(?:[.\-_]|/|$)")
is_conf_like     = path_l.str.contains(r"(?:^|/)(?:conf)(?:[.\-_]|/|$)")
is_admin_like    = path_l.str.contains(r"(?:^|/)(?:admin)(?:/|$)")
is_login_like    = path_l.str.contains(r"(?:^|/)(?:login|signin|auth)(?:[/?#]|$)")
is_ckeditor_like = path_l.str.contains(r"(?:^|/)(?:ckeditor)(?:/|$)")
is_php           = path_l.str.endswith(".php")
has_semicolon_at = path.str.contains(r"[;@]")
# random-looking token before .php (e.g., ABC123.php) with length >=6
has_random_token_php = path.str.contains(r"/[A-Za-z0-9]{6,}\.php(?:$|\?)")

# path depth (count of slashes ignoring leading/trailing)
path_depth = path.str.replace(r"^/+", "", regex=True).str.replace(r"/+$", "", regex=True).str.count("/").fillna(0).astype("int16")

# Collect all path-derived numeric features as 0/1 (or counts)
path_num_frame = pd.DataFrame({
    "uri_len": uri_len,
    "path_depth": path_depth,
    "is_config_like":   is_config_like.astype("int8"),
    "is_settings_like": is_settings_like.astype("int8"),
    "is_db_like":       is_db_like.astype("int8"),
    "is_conf_like":     is_conf_like.astype("int8"),
    "is_admin_like":    is_admin_like.astype("int8"),
    "is_login_like":    is_login_like.astype("int8"),
    "is_ckeditor_like": is_ckeditor_like.astype("int8"),
    "is_php":           is_php.astype("int8"),
    "has_semicolon_at": has_semicolon_at.astype("int8"),
    "has_random_token_php": has_random_token_php.astype("int8"),
}, index=df.index)

# -----------------------------
# 4) Per-IP rate features (1s & 5s buckets)
# -----------------------------
src_ip = df["source.ip"].fillna("").astype(str)

# safe maps (avoid NaT/NaN by casting to str)
sec_key   = src_ip + "§" + sec_bucket.astype(str)
bucket5_key = src_ip + "§" + bucket5.astype(str)

sec_counts = sec_key.value_counts()
b5_counts  = bucket5_key.value_counts()

ip_rate_1s = sec_key.map(sec_counts).fillna(0).astype("int32")
ip_rate_5s = bucket5_key.map(b5_counts).fillna(0).astype("int32")

# Unique paths per IP-second (helps catching scans)
combo_ip_sec = src_ip + "§" + sec_bucket.astype(str)
uniq_path_per_ip_sec = (path.groupby(combo_ip_sec).transform("nunique")).fillna(0).astype("int32")

rate_frame = pd.DataFrame({
    "ip_rate_1s": ip_rate_1s,
    "ip_rate_5s": ip_rate_5s,
    "uniq_paths_ip_1s": uniq_path_per_ip_sec,
}, index=df.index)

# -----------------------------
# 5) Categorical features
# -----------------------------
ua_group = ua_group_from(df["user_agent.original"])
method   = df["http.request.method"].fillna("UNKNOWN").astype(str).str.upper()

cat_frame = pd.DataFrame({
    "method": method,
    "ua_group": ua_group,
}, index=df.index)

# -----------------------------
# 6) Assemble numeric frame
# -----------------------------
num_base = pd.DataFrame({
    "status_class": status_class,
    "response_bytes": response_bytes,   # (down-weight later)
    "dest_port": dest_port_num,
    "http_version": ver,
}, index=df.index)

# final numeric set = base + rates + path-derived
num_frame = pd.concat([num_base, rate_frame, path_num_frame], axis=1)

# -----------------------------
# 7) Encode/scale
# -----------------------------
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False, dtype=np.float32)
X_cat = ohe.fit_transform(cat_frame)
cat_cols = ohe.get_feature_names_out(cat_frame.columns).tolist()

scaler = RobustScaler()
X_num = scaler.fit_transform(num_frame.astype(np.float32))

# -----------------------------
# 8) Down-weight path-derived & response_bytes to 1/3
# -----------------------------
# Build multipliers (1.0 default)
num_cols = num_frame.columns.tolist()
num_multipliers = np.ones(len(num_cols), dtype=np.float32)

# response_bytes → 1/3
if "response_bytes" in num_cols:
    num_multipliers[num_cols.index("response_bytes")] = 1.0/3.0

# ALL path-derived numerics → 1/3
path_cols = path_num_frame.columns.tolist()
for c in path_cols:
    if c in num_cols:
        num_multipliers[num_cols.index(c)] = 1.0/3.0

# apply multipliers after scaling
X_num = X_num * num_multipliers

# -----------------------------
# 9) Final matrix & save
# -----------------------------
X = np.hstack([X_num.astype(np.float32), X_cat.astype(np.float32)]).astype(np.float32)
feature_names = num_cols + cat_cols

print("✅ Feature matrix built")
print("   Rows:", X.shape[0], "  Cols:", X.shape[1])
print("🧱 Numerics:", num_cols[:12], ("... (+%d more)" % (len(num_cols)-12) if len(num_cols)>12 else ""))
print("🏷️ Categorical levels:", cat_cols[:12], ("... (+%d more)" % (len(cat_cols)-12) if len(cat_cols)>12 else ""))

# --- Save lightweight artifacts for next cells
ART_DIR = Path("/content/artifacts") if Path("/content").exists() else Path("./artifacts")
ART_DIR.mkdir(exist_ok=True, parents=True)

np.save(ART_DIR / "X.npy", X)
num_frame.to_parquet(ART_DIR / "numerics.parquet")
pd.DataFrame({"col": feature_names}).to_parquet(ART_DIR / "feature_names.parquet")
joblib.dump(ohe, ART_DIR / "ohe.joblib")
joblib.dump(scaler, ART_DIR / "scaler.joblib")

# Save multipliers as metadata (so Cells 4–8 know what's been down-weighted)
weights_meta = {
    "down_weight_to": 1/3,
    "response_bytes": True,
    "path_numeric_columns": path_cols,
    "all_numeric_columns": num_cols,
}
with open(ART_DIR / "feature_weights.json", "w") as f:
    json.dump(weights_meta, f, indent=2)

# --- show 5 head / 5 tail of a compact preview (not the huge X)
preview = pd.concat([num_frame.head(5), cat_frame.head(5)], axis=1)
print("\n🔹 Preview head (5 rows):")
display(preview)
preview_t = pd.concat([num_frame.tail(5), cat_frame.tail(5)], axis=1)
print("\n🔹 Preview tail (5 rows):")
display(preview_t)
print(f"\n📁 Artifacts saved under: {ART_DIR}")


/tmp/ipython-input-2704109713.py:59: FutureWarning: Series.view is deprecated and will be removed in a future version. Use ``astype`` as an alternative to change the dtype.
  sec_bucket = (ts.view('int64') // 10**9).astype('Int64')  # epoch seconds (nullable)
/tmp/ipython-input-2704109713.py:60: FutureWarning: Series.view is deprecated and will be removed in a future version. Use ``astype`` as an alternative to change the dtype.
  bucket5    = ((ts.view('int64') // 10**9) // 5).astype('Int64')


✅ Feature matrix built
   Rows: 1719213   Cols: 54
🧱 Numerics: ['status_class', 'response_bytes', 'dest_port', 'http_version', 'ip_rate_1s', 'ip_rate_5s', 'uniq_paths_ip_1s', 'uri_len', 'path_depth', 'is_config_like', 'is_settings_like', 'is_db_like'] ... (+7 more)
🏷️ Categorical levels: ['method_*J.R_', 'method_-', 'method_.S7C', 'method_0+BF', 'method_3OQ', 'method_9&_!', 'method_CLTI%', 'method_CONNECT', 'method_DELETE', 'method_EV5', 'method_F9_', 'method_F~L'] ... (+23 more)

🔹 Preview head (5 rows):


,status_class,response_bytes,dest_port,http_version,ip_rate_1s,ip_rate_5s,uniq_paths_ip_1s,uri_len,path_depth,is_config_like,...,is_db_like,is_conf_like,is_admin_like,is_login_like,is_ckeditor_like,is_php,has_semicolon_at,has_random_token_php,method,ua_group
0,2,88,8080,1.1,3,16,1,4,0,0,...,0,0,0,0,0,0,0,0,POST,browser
1,2,88,8080,1.1,3,17,1,4,0,0,...,0,0,0,0,0,0,0,0,POST,browser
2,2,88,8080,1.1,6,30,1,4,0,0,...,0,0,0,0,0,0,0,0,POST,browser
3,2,88,8080,1.1,6,30,1,4,0,0,...,0,0,0,0,0,0,0,0,POST,browser
4,2,88,8080,1.1,3,16,1,4,0,0,...,0,0,0,0,0,0,0,0,POST,browser



🔹 Preview tail (5 rows):


,status_class,response_bytes,dest_port,http_version,ip_rate_1s,ip_rate_5s,uniq_paths_ip_1s,uri_len,path_depth,is_config_like,...,is_db_like,is_conf_like,is_admin_like,is_login_like,is_ckeditor_like,is_php,has_semicolon_at,has_random_token_php,method,ua_group
1719208,0,0,80,0.0,18,26,1,1,0,0,...,0,0,0,0,0,0,0,0,POST,browser
1719209,0,0,80,0.0,4,26,1,1,0,0,...,0,0,0,0,0,0,0,0,POST,browser
1719210,0,0,80,0.0,4,26,1,1,0,0,...,0,0,0,0,0,0,0,0,POST,browser
1719211,0,0,80,0.0,4,26,1,1,0,0,...,0,0,0,0,0,0,0,0,POST,browser
1719212,0,0,80,0.0,4,26,1,1,0,0,...,0,0,0,0,0,0,0,0,POST,browser



📁 Artifacts saved under: /content/artifacts


In [10]:
# 🧱 Cell 4 — Autoencoder training + scoring (weights already applied in Cell 3)
# !pip -q install tensorflow==2.* joblib pyarrow

import numpy as np, pandas as pd, joblib, random, tensorflow as tf, os
from pathlib import Path

SEED = 42
np.random.seed(SEED); random.seed(SEED); tf.random.set_seed(SEED)

assert 'DATA_CSV' in globals() and DATA_CSV is not None, "Run Cell 1 first."
ART_DIR = Path("/content/artifacts") if Path("/content").exists() else Path("./artifacts")

# --- Load feature matrix and names from Cell 3
X = np.load(ART_DIR / "X.npy").astype(np.float32)
feat_names = pd.read_parquet(ART_DIR / "feature_names.parquet")["col"].tolist()
print(f"X shape: {X.shape}")

# --- Build a compact AE
inp_dim = X.shape[1]
inputs = tf.keras.Input(shape=(inp_dim,))
x = tf.keras.layers.Dense(128, activation="relu")(inputs)
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.Dense(64, activation="relu")(x)
x = tf.keras.layers.Dense(32, activation="relu")(x)
x = tf.keras.layers.Dense(64, activation="relu")(x)
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.Dense(128, activation="relu")(x)
outputs = tf.keras.layers.Dense(inp_dim, activation="linear")(x)
ae = tf.keras.Model(inputs, outputs)

# از شیء loss استفاده می‌کنیم تا مشکل load('mse') پیش نیاد
loss_fn = tf.keras.losses.MeanSquaredError(name="mse")
ae.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss=loss_fn)

cb = [tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)]
hist = ae.fit(X, X, epochs=50, batch_size=512, validation_split=0.1, shuffle=True, callbacks=cb, verbose=1)

# --- Reconstruction error (batch)
def batched_predict(model, data, bs=2048):
    out = []
    for i in range(0, data.shape[0], bs):
        out.append(model.predict(data[i:i+bs], verbose=0))
    return np.vstack(out)

X_pred = batched_predict(ae, X, bs=2048)
mse = np.mean(np.square(X - X_pred), axis=1).astype(np.float32)

# --- Save artifacts
# فرمت امن‌تر (keras) + نسخه h5 برای سازگاری
ae.save(ART_DIR / "ae_model.keras")
try:
    ae.save(ART_DIR / "ae_model.h5")
except Exception as e:
    print("Warn: could not save .h5 ->", e)

pd.DataFrame({"row_id": np.arange(len(mse)), "score_ae": mse}).to_parquet(ART_DIR / "scores_ae.parquet")
print("✅ AE trained. Saved: ae_model.keras (+h5 if possible), scores_ae.parquet")


X shape: (1719213, 54)
Epoch 1/50
3023/3023 ━━━━━━━━━━━━━━━━━━━━ 52s 14ms/step - loss: 296828096.0000 - val_loss: 6013038.5000
Epoch 2/50
3023/3023 ━━━━━━━━━━━━━━━━━━━━ 28s 9ms/step - loss: 289523712.0000 - val_loss: 4155457.5000
Epoch 3/50
3023/3023 ━━━━━━━━━━━━━━━━━━━━ 29s 10ms/step - loss: 283767744.0000 - val_loss: 4006481.5000
Epoch 4/50
3023/3023 ━━━━━━━━━━━━━━━━━━━━ 28s 9ms/step - loss: 278158784.0000 - val_loss: 4388106.5000
Epoch 5/50
3023/3023 ━━━━━━━━━━━━━━━━━━━━ 28s 9ms/step - loss: 272491040.0000 - val_loss: 5188082.0000
Epoch 6/50
3023/3023 ━━━━━━━━━━━━━━━━━━━━ 28s 9ms/step - loss: 265219232.0000 - val_loss: 17831100.0000
Epoch 7/50
3023/3023 ━━━━━━━━━━━━━━━━━━━━ 29s 10ms/step - loss: 256835232.0000 - val_loss: 4178162.2500
Epoch 8/50
3023/3023 ━━━━━━━━━━━━━━━━━━━━ 27s 9ms/step - loss: 250681184.0000 - val_loss: 6972915.5000


✅ AE trained. Saved: ae_model.keras (+h5 if possible), scores_ae.parquet


In [11]:
# 🧱 Cell 5 — Isolation Forest training + scoring (weights already applied in Cell 3)
# !pip -q install scikit-learn joblib pyarrow

import numpy as np, pandas as pd, joblib, random
from pathlib import Path
from sklearn.ensemble import IsolationForest

SEED = 42
np.random.seed(SEED); random.seed(SEED)

assert 'DATA_CSV' in globals() and DATA_CSV is not None, "Run Cell 1 first."
ART_DIR = Path("/content/artifacts") if Path("/content").exists() else Path("./artifacts")

# --- Load X
X = np.load(ART_DIR / "X.npy").astype(np.float32)
print(f"X shape: {X.shape}")

# --- Train IF (بدون contamination ثابت؛ کالیبراسیون در سلول 7)
if_model = IsolationForest(
    n_estimators=400, max_samples="auto",
    bootstrap=False, random_state=SEED, n_jobs=-1, warm_start=False
)
if_model.fit(X)

# --- Score: higher = more anomalous
score_if = (-if_model.score_samples(X)).astype(np.float32)

# --- Save
joblib.dump(if_model, ART_DIR / "if_model.joblib")
pd.DataFrame({"row_id": np.arange(len(score_if)), "score_if": score_if}).to_parquet(ART_DIR / "scores_if.parquet")
print("✅ IF trained. Saved: if_model.joblib, scores_if.parquet")


X shape: (1719213, 54)
✅ IF trained. Saved: if_model.joblib, scores_if.parquet


In [12]:
# 🧱 Cell 6 — FAST One-Class SVM variants (choose MODE)
# !pip -q install scikit-learn joblib pyarrow

import numpy as np, pandas as pd, joblib, random
from pathlib import Path

from sklearn.svm import OneClassSVM
from sklearn.decomposition import PCA
from sklearn.linear_model import SGDOneClassSVM

SEED = 42
np.random.seed(SEED); random.seed(SEED)

# ========== CONFIG ==========
MODE = "subsample_ocsvm"   # options: "subsample_ocsvm", "pca_ocsvm", "sgd_oneclass"
SUBSAMPLE_N = 50000        # for subsample_ocsvm
PCA_N = 64                 # for pca_ocsvm
NU = 0.05                  # ↑ NU => حساس‌تر (FN کمتر، FP بیشتر)
CACHE_MB = 1024            # libsvm cache
TOL = 1e-2                 # libsvm tolerance
MAX_ITER = 2000            # iteration guard
# ===========================

assert 'DATA_CSV' in globals() and DATA_CSV is not None, "Run Cell 1 first."
ART_DIR = Path("/content/artifacts") if Path("/content").exists() else Path("./artifacts")

# --- Load X
X = np.load(ART_DIR / "X.npy").astype(np.float32)
print(f"Data ready: X shape = {X.shape}, MODE = {MODE}")

def fit_score_ocsvm_on(X_train, X_score, nu=NU):
    """Train OCSVM (rbf) on X_train quickly, then score X_score."""
    svm = OneClassSVM(kernel="rbf", gamma="scale", nu=nu,
                      shrinking=True, cache_size=CACHE_MB, tol=TOL, max_iter=MAX_ITER)
    svm.fit(X_train)
    score = (-svm.decision_function(X_score)).astype(np.float32)  # higher = more anomalous
    return svm, score

if MODE == "subsample_ocsvm":
    # ---- Subsample for training; score the full matrix
    rs = np.random.RandomState(SEED)
    n_fit = min(SUBSAMPLE_N, X.shape[0])
    idx = rs.choice(X.shape[0], size=n_fit, replace=False)
    print(f"Training on subsample: {n_fit} rows (of {X.shape[0]}) ...")
    svm_model, score_svm = fit_score_ocsvm_on(X[idx], X, nu=NU)

elif MODE == "pca_ocsvm":
    # ---- PCA to reduce dimensionality then OCSVM
    print(f"PCA to {PCA_N} components...")
    pca = PCA(n_components=min(PCA_N, X.shape[1]), random_state=SEED)
    X_pca = pca.fit_transform(X).astype(np.float32)
    print("Training OCSVM (rbf) on PCA-reduced data...")
    svm_model, score_svm = fit_score_ocsvm_on(X_pca, X_pca, nu=NU)
    # save PCA for reproducibility
    joblib.dump(pca, ART_DIR / "pca_for_ocsvm.joblib")

elif MODE == "sgd_oneclass":
    # ---- Very fast linear one-class SVM
    print("Training SGDOneClassSVM (linear)...")
    sgd = SGDOneClassSVM(nu=NU, shuffle=True, random_state=SEED,
                         learning_rate="optimal", max_iter=MAX_ITER, tol=1e-3)
    sgd.fit(X)
    score_svm = (-sgd.decision_function(X)).astype(np.float32)
    svm_model = sgd
else:
    raise ValueError("Invalid MODE. Choose from: subsample_ocsvm, pca_ocsvm, sgd_oneclass")

# --- Save model + scores
model_path = {
    "subsample_ocsvm": "ocsvm_model.joblib",
    "pca_ocsvm":       "ocsvm_model_pca.joblib",
    "sgd_oneclass":    "ocsvm_model_sgd.joblib",
}[MODE]

joblib.dump(svm_model, ART_DIR / model_path)
pd.DataFrame({"row_id": np.arange(len(score_svm)), "score_svm": score_svm}).to_parquet(ART_DIR / "scores_svm.parquet")

print(f"✅ Fast SVM done. Saved model: {model_path}, scores: scores_svm.parquet")
print(f"Score stats — min: {score_svm.min():.6f}, median: {np.median(score_svm):.6f}, max: {score_svm.max():.6f}")


Data ready: X shape = (1719213, 54), MODE = subsample_ocsvm
Training on subsample: 50000 rows (of 1719213) ...


/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:305: ConvergenceWarning: Solver terminated early (max_iter=2000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


✅ Fast SVM done. Saved model: ocsvm_model.joblib, scores: scores_svm.parquet
Score stats — min: -17.884129, median: -0.000005, max: 2111.758545


In [17]:
# 🧱 Cell 7 — Ensemble (recall-first), nonlinear severity (gamma), full reports & downloads
# نیازمندی: خروجی‌های سلول‌های 4-5-6 و آرتیفکت‌های سلول 3 در ART_DIR موجود باشند.

import numpy as np, pandas as pd, os, json
from pathlib import Path

# ---- مسیر آرتیفکت‌ها
ART_DIR = Path("/content/artifacts") if Path("/content").exists() else Path("./artifacts")
assert (ART_DIR / "scores_ae.parquet").exists(), "scores_ae.parquet not found (run Cell 4)."
assert (ART_DIR / "scores_if.parquet").exists(), "scores_if.parquet not found (run Cell 5)."
assert (ART_DIR / "scores_svm.parquet").exists(), "scores_svm.parquet not found (run Cell 6)."
assert 'DATA_CSV' in globals() and DATA_CSV is not None, "Run Cell 1 to set DATA_CSV."

# ---- بارگذاری امتیازهای مدل‌ها
df_ae  = pd.read_parquet(ART_DIR / "scores_ae.parquet")
df_if  = pd.read_parquet(ART_DIR / "scores_if.parquet")
df_svm = pd.read_parquet(ART_DIR / "scores_svm.parquet")
scores = df_ae.merge(df_if, on="row_id").merge(df_svm, on="row_id")
N = len(scores); assert N > 0

# ---- صدک‌های هر مدل (بزرگ‌تر = ناهنجارتر) + کمی نویز برای شکستن تساوی‌ها
rng = np.random.default_rng(42)
for col in ["score_ae", "score_if", "score_svm"]:
    vals = scores[col].to_numpy() + rng.normal(0, 1e-9, N).astype(np.float32)
    ranks = vals.argsort().argsort().astype(np.int64)  # 0..N-1
    scores[f"p_{col.split('_')[1]}"] = (ranks + 1) / N

# ---- Ensemble حداقل FN: حداکثر صدک بین مدل‌ها
scores["p_ens"] = scores[["p_ae","p_if","p_svm"]].max(axis=1)

# ---- پرچم زدن پوزیتیو هر مدل و اجماع
Q_POS, Q_99 = 0.995, 0.999    # ~0.5% مثبت، ~0.1% = 99
scores["pos_ae"]  = scores["p_ae"]  >= Q_POS
scores["pos_if"]  = scores["p_if"]  >= Q_POS
scores["pos_svm"] = scores["p_svm"] >= Q_POS
scores["consensus3"] = scores[["pos_ae","pos_if","pos_svm"]].sum(axis=1) == 3
scores["consensus2p"]= scores[["pos_ae","pos_if","pos_svm"]].sum(axis=1) >= 2

# ---- شدت (Severity) با نگاشت غیرخطی (توانی) برای تمرکز روی مقادیر بالا
GAMMA = 3.0
p = scores["p_ens"].to_numpy()
severity = np.zeros_like(p, dtype=np.int16)
mask_99 = p >= Q_99
severity[mask_99] = 99
mask_pos = (p >= Q_POS) & (~mask_99)
if (Q_99 > Q_POS) and mask_pos.any():
    t = (p[mask_pos] - Q_POS) / (Q_99 - Q_POS)
    t = np.clip(t, 0, 1 - 1e-12)
    sev = 1 + 98 * (t ** GAMMA)   # 1..≈99
    severity[mask_pos] = np.floor(sev).astype(np.int16)
scores["severity"] = severity

print(f"Constraint check: {(severity>0).mean()*100:.4f}% >0 (target ~0.5%), {(severity==99).mean()*100:.4f}% ==99 (target ~0.1%).")

# ---- اتصال به کل ستون‌های خام برای گزارش
df_raw = pd.read_csv(DATA_CSV, dtype=str, engine="python", on_bad_lines="skip")
df_raw = df_raw.reset_index(drop=True)
scores = scores.reset_index(drop=True)
assert len(df_raw) == len(scores), "Row count mismatch vs raw CSV."

out_full = pd.concat([scores, df_raw], axis=1)

# ---- مرتب‌سازی و خروجی‌ها
out_full = out_full.sort_values(["severity","p_ens"], ascending=[False, False]).reset_index(drop=True)
out_pos  = out_full[out_full["severity"] > 0].reset_index(drop=True)

# ذخیره فایل‌ها
out_full_path_parquet = ART_DIR / "ensemble_full.parquet"
out_full_path_csv     = ART_DIR / "ensemble_full.csv"
out_pos_path_parquet  = ART_DIR / "ensemble_positives.parquet"
out_pos_path_csv      = ART_DIR / "ensemble_positives.csv"

out_full.to_parquet(out_full_path_parquet, index=False)
out_full.to_csv(out_full_path_csv, index=False)
out_pos.to_parquet(out_pos_path_parquet, index=False)
out_pos.to_csv(out_pos_path_csv, index=False)

print("✅ Saved:")
print(" -", out_full_path_parquet)
print(" -", out_full_path_csv)
print(" -", out_pos_path_parquet)
print(" -", out_pos_path_csv)

# ---- دانلود در کولب (اگر در Colab هستیم)
try:
    from google.colab import files
    print("📥 Downloading CSVs ...")
    files.download(str(out_full_path_csv))
    files.download(str(out_pos_path_csv))
except Exception as e:
    print("ℹ️ Not in Colab or download failed. Files are saved under:", ART_DIR)


Constraint check: 0.9332% >0 (target ~0.5%), 0.2687% ==99 (target ~0.1%).
✅ Saved:
 - /content/artifacts/ensemble_full.parquet
 - /content/artifacts/ensemble_full.csv
 - /content/artifacts/ensemble_positives.parquet
 - /content/artifacts/ensemble_positives.csv
📥 Downloading CSVs ...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [23]:
# 🧱 Cell 8 — Score new ZIP/CSV (isolate each upload) + dynamic feature alignment + downloads

import os, io, zipfile, re, shutil, time
import numpy as np, pandas as pd, joblib
from pathlib import Path
from typing import Optional
from sklearn.preprocessing import OneHotEncoder, RobustScaler
import tensorflow as tf

# ---------- Artifacts ----------
ART_DIR = Path("/content/artifacts") if Path("/content").exists() else Path("./artifacts")
assert (ART_DIR / "ohe.joblib").exists(), "ohe.joblib not found (run Cell 3)."
assert (ART_DIR / "scaler.joblib").exists(), "scaler.joblib not found (run Cell 3)."
assert (ART_DIR / "ae_model.h5").exists(), "ae_model.h5 not found (run Cell 4)."
assert (ART_DIR / "if_model.joblib").exists(), "if_model.joblib not found (run Cell 5)."
svm_candidates = ["ocsvm_model.joblib","ocsvm_model_pca.joblib","ocsvm_model_sgd.joblib"]
svm_path = next((ART_DIR/c for c in svm_candidates if (ART_DIR/c).exists()), None)
assert svm_path is not None, "No OCSVM model found (run Cell 6)."

Q_POS, Q_99 = 0.995, 0.999   # ~0.5% >0 | ~0.1% ==99
GAMMA = 3.0                  # تاکید روی ته‌دم (severity های بالا)

# ---------- IO Dirs ----------
BASE_DIR = Path("/content") if Path("/content").exists() else Path(".")
RAW_DIR  = BASE_DIR / "new_raw"; RAW_DIR.mkdir(exist_ok=True, parents=True)
OUT_DIR  = BASE_DIR / "new_data"; OUT_DIR.mkdir(exist_ok=True, parents=True)

# پاک‌سازی هدف‌های خروجی هر بار
for p in ["dataset_new.csv"]:
    fp = OUT_DIR / p
    if fp.exists():
        fp.unlink()

# ---------- Upload ----------
def colab_upload() -> Optional[str]:
    try:
        from google.colab import files
        print("📤 Upload .zip or .csv ...")
        up = files.upload()
        if not up: return None
        fname = list(up.keys())[0]
        dst = RAW_DIR / fname
        with open(dst, "wb") as f: f.write(up[fname])
        print(f"✅ Uploaded: {fname} ({len(up[fname]):,} bytes) → {dst}")
        return str(dst)
    except Exception:
        print("ℹ️ Not in Colab. Put file under:", RAW_DIR)
        return None

NEW_UPLOAD = None  # اگر دستی داری: مسیر فایل. وگرنه None تا Colab prompt بیاد.
if NEW_UPLOAD is None:
    NEW_UPLOAD = colab_upload()
assert NEW_UPLOAD and Path(NEW_UPLOAD).exists(), f"File not found: {NEW_UPLOAD}"

# ---------- Stage: isolate each upload ----------
def stage_csv_isolated(pth: str) -> str:
    p = Path(pth)
    # هر بار پوشه قدیمی را کامل پاک کن تا CSVهای آپلود قبلی قاطی نشوند
    unz_root = RAW_DIR / "unzipped_new"
    if unz_root.exists():
        shutil.rmtree(unz_root, ignore_errors=True)
    unz_root.mkdir(parents=True, exist_ok=True)

    dst = OUT_DIR / "dataset_new.csv"
    if p.suffix.lower() == ".csv":
        shutil.copy2(p, dst)
        return str(dst)

    if p.suffix.lower() == ".zip":
        # زیرپوشه یکتا برای هر آپلود
        sub = unz_root / time.strftime("run_%Y%m%d_%H%M%S")
        sub.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(p, 'r') as zf:
            zf.extractall(sub)
        # فقط از همین sub بخوان، نه کل unzipped_new
        csvs = list(sub.rglob("*.csv"))
        if not csvs:
            raise RuntimeError("No CSV files found in ZIP.")
        best = max(csvs, key=lambda x: x.stat().st_size)
        shutil.copy2(best, dst)
        print(f"📦 Using CSV from this upload: {best.name} ({best.stat().st_size:,} bytes)")
        return str(dst)

    raise RuntimeError("Unsupported file type (use .zip or .csv)")

NEW_DATA_CSV = stage_csv_isolated(NEW_UPLOAD)

# ---------- Load transformers/models ----------
ohe: OneHotEncoder = joblib.load(ART_DIR / "ohe.joblib")
scaler: RobustScaler = joblib.load(ART_DIR / "scaler.joblib")
custom_objects = {"mse": tf.keras.losses.MeanSquaredError()}
ae = tf.keras.models.load_model(ART_DIR / "ae_model.h5", custom_objects=custom_objects)
if_model = joblib.load(ART_DIR / "if_model.joblib")
svm_model = joblib.load(svm_path)

# ---------- Helpers (sync with Cell 3) ----------
BASE_COLS = [
    "@timestamp","http.request.method","http.response.status_code","http.version",
    "http.response.body.bytes","user_agent.original","url.path","url.domain",
    "destination.ip","destination.port","source.ip"
]

def to_int(s: pd.Series, default=0):
    return pd.to_numeric(s.fillna("").astype(str).str.replace(r"[^\d]", "", regex=True),
                         errors="coerce").fillna(default).astype("int64")

def ua_group_from(ua: pd.Series) -> pd.Series:
    u = ua.fillna("").astype(str).str.lower()
    is_browser = u.str.contains(r"mozilla|chrome|safari|firefox|edge")
    is_bot     = u.str.contains(r"curl|python|bot|go-http-client|wget|java|requests|okhttp|axios|postmanruntime|telegrambot|slackbot|discord|libwww-perl")
    is_empty   = u.str.strip().eq("")
    return (
        is_browser.map({True: "browser", False: None})
        .fillna(is_bot.map({True: "bot", False: None}))
        .fillna(is_empty.map({True: "empty", False: "other"}))
    )

def make_flags_from_path(path: pd.Series) -> pd.DataFrame:
    p = path.fillna("").astype(str).str.lower()
    df = pd.DataFrame({
        "is_login_like": p.str.contains(r"\b(login|signin|auth|session)\b|/login|/signin|/auth"),
        "is_config_like": p.str.contains(r"config(\.|\b)|settings(\.|\b)|\.ini\b|\.conf\b|\.cfg\b"),
        "is_conf_like": p.str.contains(r"\.inc\b|\.inc\.php\b|settings\.inc|config\.inc"),
        "is_db_like": p.str.contains(r"\bdb(\.|\b)|database|dump|backup|\.sql\b"),
        "is_ckeditor_like": p.str.contains(r"ckeditor|/admin/ckeditor|/scripts/history"),
        "has_random_token_php": p.str.contains(r"/[A-Za-z0-9]{6,}\.php\b"),
        "has_semicolon_at": p.str.contains(r";\d{3,}@"),
        "is_admin_like": p.str.contains(r"/admin(\b|/)|\badmin\b"),
        "is_php": p.str.contains(r"\.php(\?|$)"),
        "is_settings_like": p.str.contains(r"settings(\.|\b)|/settings"),
        "path_depth": p.str.replace(r"(^/|/$)", "", regex=True).str.split("/").apply(lambda segs: 0 if segs==[""] else len(segs)).astype(int),
    }, index=path.index)
    return df.astype("int32")

def rates_per_ip(ts: pd.Series, ip: pd.Series) -> pd.DataFrame:
    t = pd.to_datetime(ts, errors="coerce", utc=True).astype("int64") // 10**9
    df = pd.DataFrame({"ip": ip.fillna(""), "t": t}).reset_index()
    df = df.sort_values(["ip","t","index"], kind="mergesort")
    out_1s = np.zeros(len(df), dtype=np.int32)
    out_5s = np.zeros(len(df), dtype=np.int32)
    s1 = s5 = 0
    for i in range(len(df)):
        if i == 0 or df["ip"].iat[i] != df["ip"].iat[i-1]:
            s1 = s5 = i
        while df["t"].iat[i] - df["t"].iat[s1] > 1: s1 += 1
        while df["t"].iat[i] - df["t"].iat[s5] > 5: s5 += 1
        out_1s[i] = i - s1 + 1
        out_5s[i] = i - s5 + 1
    tmp = pd.DataFrame({"index": df["index"], "ip_rate_1s": out_1s, "ip_rate_5s": out_5s})
    tmp = tmp.sort_values("index").drop(columns=["index"]).reset_index(drop=True)
    return tmp

def uniq_paths_ip_1s(ts: pd.Series, ip: pd.Series, path: pd.Series) -> pd.Series:
    t = pd.to_datetime(ts, errors="coerce", utc=True).astype("int64") // 10**9
    g = pd.DataFrame({"ip": ip.fillna(""), "sec": t, "path": path.fillna("").astype(str)})
    u = g.groupby(["ip","sec"])["path"].transform("nunique").astype("int32")
    return u

WEIGHTS = {
    "status_class": 1.0,
    "response_bytes": 1/3,
    "dest_port": 1.0,
    "uri_len": 1/3,
    "http_version": 1.0,
    "hour_sin": 1.0,
    "hour_cos": 1.0,
    "hour_missing": 1.0,
    "ip_rate_1s": 1.0,
    "ip_rate_5s": 1.0,
}

# ---------- Read new CSV ----------
df_new = pd.read_csv(NEW_DATA_CSV, dtype=str, engine="python", on_bad_lines="skip").reset_index(drop=True)
missing = [c for c in BASE_COLS if c not in df_new.columns]
if missing: raise ValueError(f"Missing expected columns in new CSV: {missing}")

# ---------- Build features ----------
status_code_num  = to_int(df_new["http.response.status_code"])
status_class     = (status_code_num // 100).astype("int16")
response_bytes   = to_int(df_new["http.response.body.bytes"])
dest_port_num    = to_int(df_new["destination.port"])
http_ver         = pd.to_numeric(df_new["http.version"].fillna("").str.extract(r"(\d+(?:\.\d+)?)", expand=False),
                                 errors="coerce").fillna(0.0).astype("float32")
uri_len          = df_new["url.path"].fillna("").astype(str).str.len().astype("int32")

ts = pd.to_datetime(df_new["@timestamp"], errors="coerce", utc=True)
hour = ts.dt.hour.fillna(-1).astype("int16"); hour_eff = hour.mask(hour < 0, 0)
hour_sin = np.sin(2*np.pi*hour_eff/24).astype("float32")
hour_cos = np.cos(2*np.pi*hour_eff/24).astype("float32")
hour_missing = (hour < 0).astype("int8")

rates = rates_per_ip(df_new["@timestamp"], df_new["source.ip"])
ip_rate_1s = rates["ip_rate_1s"].astype("int32")
ip_rate_5s = rates["ip_rate_5s"].astype("int32")

flags = make_flags_from_path(df_new["url.path"])
uniq_paths = uniq_paths_ip_1s(df_new["@timestamp"], df_new["source.ip"], df_new["url.path"]).rename("uniq_paths_ip_1s").astype("int32")

num_frame_new = pd.DataFrame({
    "status_class": status_class * WEIGHTS["status_class"],
    "response_bytes": response_bytes * WEIGHTS["response_bytes"],
    "dest_port": dest_port_num * WEIGHTS["dest_port"],
    "uri_len": uri_len * WEIGHTS["uri_len"],
    "http_version": http_ver * WEIGHTS["http_version"],
    "hour_sin": hour_sin * WEIGHTS["hour_sin"],
    "hour_cos": hour_cos * WEIGHTS["hour_cos"],
    "hour_missing": hour_missing * WEIGHTS["hour_missing"],
    "ip_rate_1s": ip_rate_1s * WEIGHTS["ip_rate_1s"],
    "ip_rate_5s": ip_rate_5s * WEIGHTS["ip_rate_5s"],
}, index=df_new.index)

num_frame_new = pd.concat([num_frame_new, flags, uniq_paths], axis=1)

method   = df_new["http.request.method"].fillna("UNKNOWN").astype(str).str.upper()
ua_group = ua_group_from(df_new["user_agent.original"])
cat_frame_new = pd.DataFrame({"method": method, "ua_group": ua_group}, index=df_new.index)

# ---------- ALIGN to fitted transformers ----------
num_expected = list(getattr(scaler, "feature_names_in_", []))
if num_expected:
    num_frame_aligned = pd.DataFrame(index=num_frame_new.index)
    for c in num_expected:
        num_frame_aligned[c] = num_frame_new[c] if c in num_frame_new.columns else 0
    num_frame_aligned = num_frame_aligned.astype(np.float32)
else:
    num_frame_aligned = num_frame_new.astype(np.float32)

cat_expected = list(getattr(ohe, "feature_names_in_", []))
if cat_expected:
    fallback = {"method": "UNKNOWN", "ua_group": "other"}
    cat_frame_aligned = pd.DataFrame(index=cat_frame_new.index)
    for c in cat_expected:
        cat_frame_aligned[c] = cat_frame_new[c] if c in cat_frame_new.columns else fallback.get(c, "other")
else:
    cat_frame_aligned = cat_frame_new.copy()

# ---------- Transform ----------
X_num_new = scaler.transform(num_frame_aligned)
X_cat_new = ohe.transform(cat_frame_aligned)
X_new = np.hstack([X_num_new, X_cat_new]).astype(np.float32)

# صفر کردن اثر uri_len
feat_names = pd.read_parquet(ART_DIR / "feature_names.parquet")["col"].tolist()
if "uri_len" in feat_names:
    uri_idx = feat_names.index("uri_len")
    X_new[:, uri_idx] = 0.0

# ---------- Score ----------
X_pred = ae.predict(X_new, batch_size=1024, verbose=0)
score_ae_new = np.mean(np.square(X_new - X_pred), axis=1).astype(np.float32)
score_if_new = (-if_model.score_samples(X_new)).astype(np.float32)
try:
    score_svm_new = (-svm_model.decision_function(X_new)).astype(np.float32)
except Exception:
    score_svm_new = (-svm_model.score_samples(X_new)).astype(np.float32)

scores_new = pd.DataFrame({
    "row_id": np.arange(len(df_new)),
    "score_ae": score_ae_new,
    "score_if": score_if_new,
    "score_svm": score_svm_new,
})

# ---------- Percentiles + Ensemble + Severity ----------
N = len(scores_new)
rng = np.random.default_rng(42)
for col in ["score_ae","score_if","score_svm"]:
    vals = scores_new[col].to_numpy() + rng.normal(0, 1e-9, N).astype(np.float32)
    ranks = vals.argsort().argsort().astype(np.int64)
    scores_new[f"p_{col.split('_')[1]}"] = (ranks + 1) / N

scores_new["p_ens"] = scores_new[["p_ae","p_if","p_svm"]].max(axis=1)
scores_new["pos_ae"]  = scores_new["p_ae"]  >= Q_POS
scores_new["pos_if"]  = scores_new["p_if"]  >= Q_POS
scores_new["pos_svm"] = scores_new["p_svm"] >= Q_POS
scores_new["consensus3"] = scores_new[["pos_ae","pos_if","pos_svm"]].sum(axis=1) == 3
scores_new["consensus2p"]= scores_new[["pos_ae","pos_if","pos_svm"]].sum(axis=1) >= 2

p = scores_new["p_ens"].to_numpy()
severity = np.zeros_like(p, dtype=np.int16)
mask_99 = p >= Q_99
severity[mask_99] = 99
mask_pos = (p >= Q_POS) & (~mask_99)
if (Q_99 > Q_POS) and mask_pos.any():
    t = (p[mask_pos] - Q_POS) / (Q_99 - Q_POS)
    t = np.clip(t, 0, 1 - 1e-12)
    sev = 1 + 98 * (t ** GAMMA)
    severity[mask_pos] = np.floor(sev).astype(np.int16)
scores_new["severity"] = severity
print(f"NEW Stats: {(severity>0).mean()*100:.4f}% >0 | {(severity==99).mean()*100:.4f}% ==99")

# ---------- Save & Download (full + positives; with ALL raw columns) ----------
out_full_new = pd.concat([scores_new, df_new.reset_index(drop=True)], axis=1)
out_full_new = out_full_new.sort_values(["severity","p_ens"], ascending=[False, False]).reset_index(drop=True)
out_pos_new  = out_full_new[out_full_new["severity"]>0].reset_index(drop=True)

out_full_csv = ART_DIR / "NEW_full.csv"
out_pos_csv  = ART_DIR / "NEW_positives.csv"
out_full_parq= ART_DIR / "NEW_full.parquet"
out_pos_parq = ART_DIR / "NEW_positives.parquet"

out_full_new.to_csv(out_full_csv, index=False)
out_pos_new.to_csv(out_pos_csv, index=False)
out_full_new.to_parquet(out_full_parq, index=False)
out_pos_new.to_parquet(out_pos_parq, index=False)

print("✅ Saved:")
print(" -", out_full_csv)
print(" -", out_pos_csv)
print(" -", out_full_parq)
print(" -", out_pos_parq)

try:
    from google.colab import files
    print("📥 Downloading CSVs ...")
    files.download(str(out_full_csv))
    files.download(str(out_pos_csv))
except Exception:
    print("ℹ️ Not in Colab or download blocked. Files are under:", ART_DIR)


📤 Upload .zip or .csv ...


Saving Untitled Discover session (30).zip to Untitled Discover session (30) (2).zip
✅ Uploaded: Untitled Discover session (30) (2).zip (394,960 bytes) → /content/new_raw/Untitled Discover session (30) (2).zip
📦 Using CSV from this upload: Untitled Discover session (30).csv (8,642,133 bytes)


/tmp/ipython-input-4286035888.py:186: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  ts = pd.to_datetime(df_new["@timestamp"], errors="coerce", utc=True)
/tmp/ipython-input-4286035888.py:136: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  t = pd.to_datetime(ts, errors="coerce", utc=True).astype("int64") // 10**9
/tmp/ipython-input-4286035888.py:121: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  "is_login_like": p.str.contains(r"\b(login|signin|auth|session)\b|/login|/signin|/auth"),
/tmp/ipython-input-4286035888.py:122: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the

NEW Stats: 0.9552% >0 | 0.1970% ==99
✅ Saved:
 - /content/artifacts/NEW_full.csv
 - /content/artifacts/NEW_positives.csv
 - /content/artifacts/NEW_full.parquet
 - /content/artifacts/NEW_positives.parquet
📥 Downloading CSVs ...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>